# B08-00 — 실제 관측 배지 조성 기반 후속 실험 후보군 설계

**BioCulture | 교육용, 회고적(offline) 의사결정 설계 | Clean v2**

B07-00~04에서 실측한 1,025개 고유 배지 조성을 활용하여 B08 후속 실험 선정의 **후보군과 검증 계약**을 만든다. 본 단계는 새로운 실험 또는 모델 기반 최적화 실행이 아니다.

### 연구 질문
1. 측정이 있는 조성은 얼마나 되는가? 2. K, r 지표가 모두 있는 조성은? 3. 라운드 간 이동과 중복을 어떻게 관리할 것인가? 4. 실제 추가 실험 전 무엇을 검증해야 하는가?

**금지되는 주장:** 관측 배지 중 성능이 높은 것을 실험적으로 검증된 최적 배지라고 부르지 않는다. 학습 전체 데이터를 이용한 후보 선택을 미래 정보가 없는 검증이라고 주장하지 않는다. 원본 K/r 정의가 명확해지기 전에 생물학적 정량 단위를 단정하지 않는다.

**Clean v2 수정:** B07 조성별 테이블에 이미 들어 있던 `n_rounds` 등의 집계 열과 B08에서 재집계한 열이 충돌하지 않도록 B07 값은 `b07_source_` 접두어로 보존한다. 실제 곡선 테이블에서 재계산한 `n_rounds`, `n_curve_records`를 공식 B08 값으로 사용한다. 두 값의 불일치도 감사용으로 남긴다.


## 1. 재현 환경 및 원본 산출물 위치

프로젝트 루트는 현재 폴더 또는 상위 폴더에서 자동 탐색한다. B07-00 Clean v2의 표준 산출물과 B07-03의 라운드 검증을 읽는다. 현재 Notebook의 출력은 `B08_00_candidate_design_v1`에만 기록하여 기존 결과를 보존한다.

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,*cwd.parents] if (p/'results'/'bioculture'/'B07_00_ml_data_audit_v2').is_dir()),None)
if ROOT is None:
    raise FileNotFoundError('B07-00 v2 결과가 있는 bioimage-ai 프로젝트 루트에서 실행하세요.')
BASE=ROOT/'results'/'bioculture'
P0=BASE/'B07_00_ml_data_audit_v2'
P1=BASE/'B07_01_medium_growth_eda_v1'
P3=BASE/'B07_03_group_round_validation_v1'
OUT=BASE/'B08_00_candidate_design_v2'
OUT.mkdir(parents=True,exist_ok=True)
PROFILE=P0/'ml_medium_profile_level_audit_table.csv'
CURVE=P0/'ml_curve_level_audit_table.csv'
FEATURE=P0/'medium_feature_audit.csv'
EDA=P1/'eda_medium_profile_table.csv'
for p in [PROFILE,CURVE,FEATURE,EDA]:
    if not p.is_file(): raise FileNotFoundError(p)
print('Root:',ROOT,'\nPython:',sys.version.split()[0],'\nOutput:',OUT)


## 2. 원자료·조성 단위·목표값 감사

배지 성분 44개는 화학적 입력값이며 `medium_profile_id`는 동일 조성을 묶는 식별자이다. `K_median`, `r_median`은 B07-01에서 계산된 조성 수준의 요약값이다. 결측 목표는 학습 및 사후 점수 계산에서 따로 취급한다.

In [ ]:
profiles=pd.read_csv(PROFILE,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
curves=pd.read_csv(CURVE,encoding='utf-8-sig',dtype={'medium_profile_id':'string','Condition ID':'string','Label':'string'},low_memory=False)
fstat=pd.read_csv(FEATURE,encoding='utf-8-sig')
eda=pd.read_csv(EDA,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
features=fstat['feature'].astype(str).tolist()
assert len(features)==44 and len(profiles)==1025 and len(curves)==13608
assert profiles.medium_profile_id.is_unique and eda.medium_profile_id.is_unique
assert set(profiles.medium_profile_id)==set(eda.medium_profile_id)
assert all(c in profiles.columns for c in features)
merged=profiles.merge(eda[['medium_profile_id','K_median','r_median']],on='medium_profile_id',how='left',validate='one_to_one',indicator=True,suffixes=('','_eda'))
assert (merged['_merge']=='both').all()
for t in ['K_median','r_median']:
    if t+'_eda' in merged.columns:
        if t in merged.columns:
            assert np.allclose(pd.to_numeric(merged[t],errors='coerce'),pd.to_numeric(merged[t+'_eda'],errors='coerce'),equal_nan=True)
            merged=merged.drop(columns=t+'_eda')
        else: merged=merged.rename(columns={t+'_eda':t})
merged=merged.drop(columns='_merge')
assert all(pd.to_numeric(merged[f],errors='coerce').notna().all() for f in features)
print('조성',len(merged),'곡선',len(curves),'성분',len(features))
print('유효 조성:',{t:int(merged[t].notna().sum()) for t in ['K_median','r_median']})


## 3. 라운드 및 반복 구조: 증거 기반 그룹화

같은 조성은 여러 원본 `Condition ID`에서 나타날 수 있다. B07-00에서는 2개 Condition ID에 서로 다른 농도가 기록된 사례가 확인되었다. B08에서도 원본 Condition ID만으로 조성을 식별하지 않는다. 

라운드가 다른 관측은 같은 생물학적 모집단의 독립 복제라고 가정하지 않는다.
**Clean v2 주의:** B07의 요약 열 이름과 B08 재계산 열 이름이 충돌하면, B07 값을 `b07_source_...`로 보존하고 B08의 실측 곡선 재집계 열을 후속 분석에 사용한다. 차이가 있으면 `audit_mismatch_...` 플래그를 남긴다.


In [ ]:
def as_round(s):
    s=str(s)
    if s.endswith('.0'): s=s[:-2]
    return s.replace('Round','').lstrip('0') or '0'

curves['round_key']=curves['round'].map(as_round)
groups=(curves.groupby('medium_profile_id',sort=True)
        .agg(n_curve_records=('Label','size'),
             n_rounds=('round_key','nunique'),
             original_condition_ids=('Condition ID','nunique'))
        .reset_index())
round_membership=(curves.groupby('medium_profile_id')['round_key']
                  .agg(lambda x: ','.join(sorted(set(x),key=int)))
                  .rename('round_membership').reset_index())
groups=groups.merge(round_membership,on='medium_profile_id',validate='one_to_one')

# B07-00 테이블에도 n_rounds 등이 있으므로 병합 전 원본 이름을 보존해 충돌을 방지한다.
# 단순 suffix 자동 부여 대신 명시적 명명으로 검증 가능성을 높인다.
audit_cols=['n_curve_records','n_rounds','original_condition_ids','round_membership']
rename_b07={c:'b07_source_'+c for c in audit_cols if c in merged.columns}
base=merged.rename(columns=rename_b07)
candidate=base.merge(groups,on='medium_profile_id',how='inner',validate='one_to_one')
assert len(candidate)==1025 and candidate.medium_profile_id.is_unique
assert 'n_rounds' in candidate.columns and 'n_curve_records' in candidate.columns
assert int(candidate['n_curve_records'].sum())==len(curves)

# B07에 저장된 기존 집계값과 새 집계값의 차이는 숨기지 않고 보고한다.
for col in audit_cols:
    previous='b07_source_'+col
    if previous in candidate.columns:
        left=candidate[previous].astype('string').fillna('<NA>')
        right=candidate[col].astype('string').fillna('<NA>')
        mismatch=left.ne(right).fillna(False)
        candidate['audit_mismatch_'+col]=mismatch
        print(f'B07 vs B08 {col} 불일치 조성 수:',int(mismatch.sum()))
        if mismatch.any():
            print(candidate.loc[mismatch,['medium_profile_id',previous,col]].head(5).to_string(index=False))

print('중복 라운드 조성:',int((candidate['n_rounds']>1).sum()))
print('관측 곡선수 중앙값:',candidate['n_curve_records'].median())
print('라운드별 관측 곡선 수:')
print(curves.groupby('round_key').size().sort_index().to_string())


## 4. 분석 가능성 등급 (수학적·실험적 검증과는 다름)

- `both_targets_available`: 원본 `K`와 `r`의 조성별 중앙값이 모두 존재한다.
- `repeated_round`: 적어도 두 라운드에서 관측된 조성이다. **재현성 입증이 아니라 후속 검토 우선 대상**이다.
- `measured_only`: 실측된 배지 조성만 포함한다. 새로운 조성에 대한 외삽 후보는 생성하지 않는다.

관측값이 많은 조성을 좋은 배지로 간주하지 않는다. 점수 최적화 목표는 아직 정하지 않는다.

In [ ]:
candidate['both_targets_available']=candidate['K_median'].notna()&candidate['r_median'].notna()
candidate['repeated_round']=candidate['n_rounds']>1
candidate['measured_only']=True
candidate['readiness_label']=np.select([candidate.both_targets_available&candidate.repeated_round,candidate.both_targets_available],['two_targets_multi_round','two_targets_single_round'],default='target_incomplete')
readiness=candidate.groupby('readiness_label',dropna=False).agg(n_profiles=('medium_profile_id','size'),n_curve_records=('n_curve_records','sum')).reset_index()
assert readiness.n_profiles.sum()==1025
print(readiness.to_string(index=False))


## 5. 관측 상위값은 기술적 예시이지 최적화 결과가 아니다

상위 `K_median` / `r_median` 배지 조성을 **이미 관측된 사례**로 기록한다. 모델 예측값을 쓰지 않는다. 라운드 교란이 있고 두 목표가 동시에 높지 않을 수 있으므로, 상위 순위만으로 실험할 조건을 확정하지 않는다. 새로운 화학 조성의 생성·추천은 B08-00 범위 밖이다.

In [ ]:
rank_rows=[]
for target in ['K_median','r_median']:
    q=candidate.loc[candidate[target].notna(),['medium_profile_id','round_membership','n_curve_records','n_rounds','K_median','r_median']].copy()
    q=q.sort_values([target,'medium_profile_id'],ascending=[False,True]).head(15)
    q.insert(0,'rank',range(1,len(q)+1));q.insert(0,'ranking_target',target)
    rank_rows.append(q)
observed_top=pd.concat(rank_rows,ignore_index=True)
assert set(observed_top.medium_profile_id).issubset(set(candidate.medium_profile_id))
print(observed_top.groupby('ranking_target').head(5).to_string(index=False))


## 6. B08 후속 오프라인 검증 프로토콜

다음 Notebook에서는 **숨겨둔 실제 측정값을 사후에만 공개**하는 모의 순차 실험을 사용한다. 

1. 조성 ID별로 관측 목표값을 가리고, 초기 관측 집합을 지정한다.
2. 나머지 이미 측정된 조성을 미측정 후보로 **가정**한다.
3. 초기 집합에서만 모델·전처리·선정 기준을 학습한다.
4. 선택된 배지 조성의 기록된 측정값을 공개해 취득 성능을 계산한다.
5. 단순 무작위 선정과 비교하고 여러 초기 집합으로 반복한다.
6. 같은 실험 라운드의 조성 구성이 결과에 미치는 영향과 결과의 적용 한계를 검사한다.

**오프라인 선택 = 새로운 wet-lab 실험 결과가 아니다.** 특히 라운드 교란을 통제할 수 없는 경우 최적화 성과를 인과적 개선으로 보고하지 않는다.

In [ ]:
protocol={
 'dataset':'Aida–Ying 2025 E. coli medium composition / observed growth summaries',
 'unit_of_selection':'medium_profile_id',
 'n_measured_profiles':int(len(candidate)),
 'features':features,
 'targets':['K_median','r_median'],
 'target_definitions_verified':False,
 'next_stage':'B08-01 retrospective acquisition benchmark',
 'no_new_experiments_performed':True,
 'anti_leakage':['split on medium_profile_id before modeling','fit preprocessing only on revealed/training pool','hide candidate outcomes until after choice','forbid globally fit transformations in CV'],
 'limits':['historical measured pool only','round and composition confounding','K/r source definitions unverified','simulation not laboratory validation']}
assert candidate.medium_profile_id.nunique()==len(candidate)
print('Offline protocol defined; actual sequential experiment simulation has NOT been run.')


## 7. 감사 산출물 저장과 재읽기

원본 Excel을 덮어쓰지 않는다. 점수 순위는 관측된 결과의 기술적 표이며 후속 실험 실행 목록이 아니다. 서명은 파일 자체의 SHA256이고 원본 데이터에 대한 생물학적 검증을 뜻하지 않는다.

In [ ]:
save_a=OUT/'candidate_pool_measured_profiles.csv'
save_b=OUT/'observed_top_profiles_descriptive.csv'
save_c=OUT/'candidate_readiness_audit.csv'
save_d=OUT/'offline_acquisition_protocol.json'
candidate.to_csv(save_a,index=False,encoding='utf-8-sig')
observed_top.to_csv(save_b,index=False,encoding='utf-8-sig')
readiness.to_csv(save_c,index=False,encoding='utf-8-sig')
save_d.write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
reloaded=pd.read_csv(save_a,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
assert len(reloaded)==1025 and reloaded.medium_profile_id.is_unique
assert int(reloaded.both_targets_available.sum())==int(candidate.both_targets_available.sum())
assert int(reloaded.repeated_round.sum())==int(candidate.repeated_round.sum())
status={'status':'B08_00_CANDIDATE_POOL_AUDITED_NOT_OPTIMIZED','n_profiles':1025,'n_features':44,
 'n_target_complete':int(candidate.both_targets_available.sum()),
 'n_multiple_rounds':int(candidate.repeated_round.sum()),
 'input_files':[str(PROFILE),str(CURVE),str(FEATURE),str(EDA)],
 'files_saved':[p.name for p in [save_a,save_b,save_c,save_d]],
 'limitations':protocol['limits']}
report=OUT/'B08_00_validation_report.json'
report.write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
    for x in iter(lambda:f.read(1048576),b''):h.update(x)
 return h.hexdigest()
paths=[PROFILE,CURVE,FEATURE,EDA,save_a,save_b,save_c,save_d,report]
manifest=pd.DataFrame([{'type':'input' if p in [PROFILE,CURVE,FEATURE,EDA] else 'output','path':str(p),'sha256':sha256(p),'size_bytes':p.stat().st_size} for p in paths])
manifest.to_csv(OUT/'file_manifest_sha256.csv',index=False,encoding='utf-8-sig')
print(json.dumps(status,ensure_ascii=False,indent=2))


## 8. 학습 연습문제 (해답은 Word 해설서에 별도 작성)

1. `Condition ID` 대신 `medium_profile_id`를 사용해야 하는 이유를 실제 사례로 설명한다.
2. `both_targets_available=True`인 조성의 수와 전체 비율을 계산한다.
3. 라운드가 두 개 이상인 조성의 목록을 만들고, 이를 독립적 재현성의 증거라고 볼 수 없는 이유를 설명한다.
4. 상위 관측값 순위를 그대로 최적화 성과라고 보고하면 생기는 문제를 기술한다.
5. 누출 없는 회고적 순차 선택 실험을 위해 초기 집합·후보 집합·관측 공개 시점을 제안한다.

**다음 순서:** 실행 Notebook 제출 → 실제 결과 검증 → B08-00 상세 Word 해설서 → B08-01 회고적 순차 선택 벤치마크.